In [1]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama , OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

In [4]:
loader=PyPDFLoader("./data/Academic-Calenda-2025.pdf")

In [5]:
docs=loader.load()

In [6]:
len(docs)

4

In [8]:
spitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
spitter_docs=spitter.split_documents(docs)

In [9]:
len(spitter_docs)

7

In [10]:
embeddings=OllamaEmbeddings(model="nomic-embed-text",
    base_url="http://localhost:11434" )

vector_store=Chroma.from_documents(
    documents=spitter_docs,
    embedding=embeddings
    )


In [14]:
@tool
def retriver_tool(query:str):
    """
      This tool retrieves information from the official University Academic Calendar.
      Use this tool whenever the user asks about semester dates, exam schedules, 
      holidays, admission deadlines, class start/end dates, or academic events.    
    """
    docs=vector_store.similarity_search(query=query,k=4)
    context=""

    for doc in docs:
        context=doc.page_content +"\n\n"

    return context

In [ ]:
llm = ChatOllama(
    model="mistral",
    temperature=0  
)

In [29]:
system_prompt = """You are a helpful and precise University Academic Assistant.

Your responsibilities:
1. Answer general greetings and common questions directly and politely without using tools.
2. For any query related to semester schedules, admission deadlines, exam dates, class routines, holidays, or official academic events, you MUST use the `retriever_tool` to fetch accurate information from the University Academic Calendar.
3. Base your academic answers strictly on the retrieved context. If the requested information is not present in the retrieved document, clearly state that the information is not found in the academic calendar.
4. Keep your responses clear, well-structured, and easy for students to read.
5. ALWAYS use 'retriver_tool' tool for questions requiring knowledge
"""

In [31]:
agent=create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=system_prompt
)

In [32]:
query="When does the mid-term exam start summer semister"
response=agent.invoke({"messages":[{"role":"user","content":query}]})


In [33]:
result=response["messages"][-1].content
print(result)

 To find the exact dates for the mid-term exams during the summer semester, I will use the `retriever_tool`.

[{"name":"retriver_tool","arguments":{"query":"mid-term exam start date summer semester"}}]

Please wait while I retrieve the information...

According to the University Academic Calendar, the mid-term exams for the summer semester are scheduled to start on [insert date here].

If you have any other questions or need further clarification, feel free to ask!
